In [1]:
import os
import copy
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import albumentations as A
import torch.nn.functional as F

import cv2
import torch
import torch.nn as nn

from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, DPTForDepthEstimation
from PIL import Image
from torch.utils.tensorboard import SummaryWriter
from sklearn.decomposition import PCA

c:\corecore\corecore\Lib\site-packages\albumentations\check_version.py:147: UserWarning: Error fetching version info <urlopen error _ssl.c:1011: The handshake operation timed out>
  data = fetch_version_info()


In [2]:
from vision_transformer import DINOHead
import utils

In [3]:
label_path = r'6 dataset (160-2560, step=140)\masks (3 classes) (new)'
srez_path = r'6 dataset (160-2560, step=140)\initial images (new)'

In [4]:
label = cv2.imread(label_path + r"\output_00160.tif")[:, :, 0]
srez = cv2.imread(srez_path + r"\00160.png")[:, :, 0]

In [5]:
files = list(zip(os.listdir(label_path), os.listdir(srez_path)))
n = len(files)

num_classes = 2
image_list = []
target_list = []
for i, (l_p, s_p) in enumerate(files):
    label = cv2.imread(label_path + '/' + l_p)[:, :, 0]
    srez = cv2.imread(srez_path + '/' + s_p)[:, :, 0]

    c_x = srez.shape[0] // 2
    c_y = srez.shape[1] // 2
    w = label.shape[0] // 2
    h = label.shape[1] // 2
    crop_srez = srez[(c_x-w):(c_x+w), (c_y-h):(c_y+h)]
    
    label[label == 2] = 0

    image_list.append(crop_srez)
    target_list.append(label)


In [6]:
class CoreDataset(torch.utils.data.Dataset):
    def __init__(self, labels, srez_list, transform=None):
        """
        labels: list of numpy arrays (H,W) или (H,W,1)
        srez_list: list of numpy arrays (H,W) или (H,W,1)
        transform: albumentations.Compose или любой кастомный трансформ
        """
        assert len(labels) == len(srez_list), "Labels and srez lists must have the same length"
        
        self.labels = labels
        self.srez_list = srez_list
        self.transform = transform
        self.processor = AutoImageProcessor.from_pretrained("facebook/dpt-dinov2-base-nyu")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = self.labels[idx].astype("float32")
        srez = self.srez_list[idx].astype("float32")

        c_x = srez.shape[0] // 2
        c_y = srez.shape[1] // 2
        w = label.shape[0] // 2
        h = label.shape[1] // 2

        crop_srez = srez[(c_x - w):(c_x + w), (c_y - h):(c_y + h)]


        if self.transform:
            transformed1 = self.transform(image=crop_srez)
            crop_srez1 = transformed1['image']
            transformed2 = self.transform(image=crop_srez)
            crop_srez2 = transformed2['image']
        else:
            crop_srez = np.expand_dims(crop_srez, axis=0)
            crop_srez = self.processor(crop_srez)['pixel_values'][0]
            return crop_srez, label

        crop_srez1 = np.expand_dims(crop_srez1, axis=0)

        crop_srez1 = self.processor(crop_srez1)['pixel_values'][0]

        crop_srez2 = np.expand_dims(crop_srez2, axis=0)

        crop_srez2 = self.processor(crop_srez2)['pixel_values'][0]


        return [crop_srez1, crop_srez2]

In [7]:
class SegmentationHead(nn.Module):
    def __init__(self, embed_dim=768, num_classes=2, img_size=518, patch_size=14):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2 
        

        self.head = nn.Sequential(

            nn.Conv2d(embed_dim, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),  # ×2
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # ×2
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, num_classes, kernel_size=1)
        )

        self._initial_state = copy.deepcopy(self.head.state_dict())
        
    def forward(self, features):
        B, N, C = features.shape
        
        if N == self.num_patches + 1:
            features = features[:, 1:, :] 
        
        patch_dim = int(self.img_size // self.patch_size) 
        features = features.permute(0, 2, 1).view(B, C, patch_dim, patch_dim)
        
        logits = self.head(features) 

        logits = F.interpolate(logits, size=(self.img_size, self.img_size), 
                               mode='bilinear', align_corners=False)
        
        return logits
    def reset(self):
        self.head.load_state_dict(self._initial_state)

In [8]:
model1 = DPTForDepthEstimation.from_pretrained("facebook/dpt-dinov2-base-nyu")
model2 = DPTForDepthEstimation.from_pretrained("facebook/dpt-dinov2-base-nyu")

student = model1.backbone
teacher = model2.backbone

validation_head = SegmentationHead()

student_HEAD = DINOHead(768, 65536)
teacher_HEAD = DINOHead(768, 65536)
for ps, pt in zip(student_HEAD.parameters(), teacher_HEAD.parameters()):
    pt.data = ps.data
    pt.requires_grad = False

for p in teacher.parameters():
    p.requires_grad = False

total = sum(p.numel() for p in student.parameters())
trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
print("Student")
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,}")

total = sum(p.numel() for p in teacher.parameters())
trainable = sum(p.numel() for p in teacher.parameters() if p.requires_grad)
print("Teacher")
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,}")

Loading weights:   0%|          | 0/281 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/281 [00:00<?, ?it/s]

Student
Total params: 86,580,480
Trainable params: 86,580,480
Teacher
Total params: 86,580,480
Trainable params: 0


c:\corecore\corecore\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


student = utils.MultiCropWrapper(student, DINOHead(
        embed_dim,
        args.out_dim,
        use_bn=args.use_bn_in_head,
        norm_last_layer=args.norm_last_layer,
    ))

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 4
num_epochs = 100
learning_rate = 1e-3
val_epochs = 5

optimizer = torch.optim.AdamW([
        {'params': student.parameters()},
        {'params': student_HEAD.parameters()}
        ],
        lr=learning_rate
    )

loss_fn = torch.nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',         
    factor=0.2,    
    patience=3,            
    min_lr=1e-6,
    threshold = 1e-3          
)

In [10]:
def compute_miou(predictions, targets, num_classes, ignore_index=255):
    preds = predictions.argmax(dim=1)  # [B, H, W]
    valid = (targets != ignore_index)
    
    # Маскируем
    preds = preds * valid + ignore_index * (~valid)
    targets = targets * valid + ignore_index * (~valid)
    
    ious = []
    for cls in range(num_classes):
        pred_cls = (preds == cls)
        target_cls = (targets == cls)
        inter = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()
        if union > 0:
            ious.append(inter / union)
    
    return sum(ious) / len(ious) if ious else 0.0

In [11]:
def crop_4(image: np.ndarray, crop_size: int = 518):
    """
    Делит одно изображение (NumPy array) на 4 кропа заданного размера.

    Args:
        image (np.ndarray): массив [H, W] или [H, W, C]
        crop_size (int): размер кропа

    Returns:
        patches (list of np.ndarray): список из 4 кропов
    """
    if image.ndim == 2:
        H, W = image.shape
        C = None
    elif image.ndim == 3:
        H, W, C = image.shape
    else:
        raise ValueError("image должен быть [H, W] или [H, W, C]")

    if H < crop_size or W < crop_size:
        raise ValueError(f"Изображение слишком маленькое ({H}x{W}) для кропа {crop_size}x{crop_size}")

    # верхний левый
    patch1 = image[0:crop_size, 0:crop_size] if C is None else image[0:crop_size, 0:crop_size, :]
    # верхний правый
    patch2 = image[0:crop_size, W-crop_size:W] if C is None else image[0:crop_size, W-crop_size:W, :]
    # нижний левый
    patch3 = image[H-crop_size:H, 0:crop_size] if C is None else image[H-crop_size:H, 0:crop_size, :]
    # нижний правый
    patch4 = image[H-crop_size:H, W-crop_size:W] if C is None else image[H-crop_size:H, W-crop_size:W, :]

    return [patch1, patch2, patch3, patch4]

In [12]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm

def plot_pca_2d(pca_2d_embeddings, labels, epoch=None, title=None, 
                connect_pairs=False, view_markers=None):
    """
    Визуализация PCA [N, 2] с цветными метками
    
    Args:
        pca_2d_embeddings: np.array [N, 2] — координаты после PCA
        labels: list/array [N] — метки (например, ID изображений)
        connect_pairs: bool — соединять ли линии между view1/view2 одного изображения
        view_markers: dict — {'v1': 'o', 'v2': 'x'} для разных маркеров
    """
    plt.figure(figsize=(8, 6))
    
    # Цветовая схема: каждая уникальная метка — свой цвет
    unique_labels = np.unique(labels)
    color_map = plt.get_cmap('tab10', len(unique_labels))
    label_to_color = {lbl: color_map(i) for i, lbl in enumerate(unique_labels)}
    
    # Рисуем точки
    for i, (x, y) in enumerate(pca_2d_embeddings):
        lbl = labels[i]
        marker = 'o'
        
        # Если есть информация о view (например, '42_v1', '42_v2')
        if view_markers and isinstance(lbl, str):
            for view_key, marker_symbol in view_markers.items():
                if view_key in lbl:
                    marker = marker_symbol
                    break
        
        plt.scatter(x, y, c=[label_to_color[lbl]], marker=marker, 
                   s=40, alpha=0.7, edgecolors='white', linewidth=0.5)
    
    # Соединяем пары одного изображения (для DINO: view1 ↔ view2)
    if connect_pairs:
        # Ожидаем метки вида '42_v1', '42_v2'
        img_to_idx = {}
        for i, lbl in enumerate(labels):
            if isinstance(lbl, str) and '_' in lbl:
                img_id = lbl.rsplit('_', 1)[0]  # '42_v1' → '42'
                if img_id not in img_to_idx:
                    img_to_idx[img_id] = []
                img_to_idx[img_id].append(i)
        
        for img_id, idx_list in img_to_idx.items():
            if len(idx_list) == 2:  # ровно 2 views
                i1, i2 = idx_list
                x1, y1 = pca_2d_embeddings[i1]
                x2, y2 = pca_2d_embeddings[i2]
                plt.plot([x1, x2], [y1, y2], 'gray', linewidth=0.8, alpha=0.3)
    
    # Оформление
    plt.xlabel('PC1', fontsize=10)
    plt.ylabel('PC2', fontsize=10)
    plt.title(title or f'PCA Visualization | Epoch {epoch}' if epoch else 'PCA Visualization', 
              fontsize=12, pad=15)
    plt.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    
    return plt.gcf()

In [ ]:
transform = A.Compose([
    A.GaussNoise(p=0.5, std_range=[0.05, 0.1]),
    A.GaussianBlur(p=0.5, sigma_limit=[1, 2]),
    A.Sharpen(p=0.5),
    A.SaltAndPepper(p=0.5, amount = [0.005, 0.01]),
    A.Solarize(p=0.2),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomCrop(height = 450, width = 450, p=1, pad_if_needed = True),
    A.Normalize(),
])

train_target = [crop_4(p) for p in target_list[:32]] 
train_image = [crop_4(p) for p in image_list[:32]] 

train_target = [item for sublist in train_target for item in sublist]
train_image = [item for sublist in train_image for item in sublist]

train_dataset = CoreDataset(train_target, train_image, transform)

val_target = [crop_4(p) for p in target_list[32:]] 
val_image = [crop_4(p) for p in image_list[32:]] 

val_target = [item for sublist in val_target for item in sublist]
val_image = [item for sublist in val_image for item in sublist]

validation_dataset = CoreDataset(val_target, val_image)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=True)

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [14]:
#гиперпараметры
l = 0.996
m = 0.9

In [15]:
def H(t, s, C, tps = 0.4, tpt = 0.04):
    t = t.detach()

    before = torch.isnan(s).any() or torch.isnan(t).any()

    s = F.softmax(s / tps, dim = 1)
    t = F.softmax((t - C) / tpt, dim = 1)
    
    if (torch.isnan(s).any() or torch.isnan(t).any()) and not(before):
        print("всё из-за софтмакса")

    # print(s.sum(), t.sum())

    # print(torch.log(s + 1e-8).sum(), (t*torch.log(s)).sum())

    result = - (t * torch.log(s + 1e-8)).sum(dim = 1).mean()
    
    if torch.isnan(result).any() and not(before):
        print('наскладывал, проверяй')

    return result

In [ ]:
student.cuda()
teacher.cuda()
student_HEAD.cuda()
teacher_HEAD.cuda()


log_dir = f"runs/dino {datetime.now().strftime('%d_%m %H_%M')}"
writer = SummaryWriter(log_dir=log_dir)

metodpisi = PCA(n_components = 2)

C = torch.zeros([65536], device = 'cuda')
for epoch in range(num_epochs):

    for p in student.parameters():
        p.requires_grad = True

    train_loss = 0
    student_embeddings = []
    inputs_fig, inputs_axes = plt.subplots(5, 2, figsize=(18, 5 * 5))

    for batch, idx in tqdm(enumerate(train_loader)):
        x1, x2 = batch[0].cuda(), batch[1].cuda()
        if idx % 5 == 0:
            inputs_axes[idx // 5 - 1, ] 

        if torch.isnan(x1).any() or torch.isnan(x2).any():
            print("альбументации руина!!!!")

        s1, s2 = student(x1).feature_maps[3], student(x2).feature_maps[3] 
        t1, t2 = teacher(x1).feature_maps[3], teacher(x2).feature_maps[3]

        s1, s2 = s1[:, 0, :], s2[:, 0, :]
        t1, t2 = t1[:, 0, :], t2[:, 0, :] 

        s1, s2 = student_HEAD(s1), student_HEAD(s2)
        t1, t2 = teacher_HEAD(t1), teacher_HEAD(t2)

        student_embeddings.append(s1.cpu().detach().view(batch_size, -1))
        student_embeddings.append(s2.cpu().detach().view(batch_size, -1))
            
        loss = H(t1, s2, C) / 2 + H(t2, s1, C) / 2

        train_loss += loss / batch_size

        loss.backward()
        
        optimizer.step()
        optimizer.zero_grad()

        for gs, gt, in zip(student.parameters(), teacher.parameters()):
            gt.data = l * gt.data + (1 - l) * gs.data
        for gs, gt, in zip(student_HEAD.parameters(), teacher_HEAD.parameters()):
            gt.data = l * gt.data + (1 - l) * gs.data
            
        C = m * C + (1 - m) * torch.cat([t1, t2], dim = 0).mean(dim = 0)

    train_loss /= len(train_loader)
    
    #Дичайшая валидация

    #отправляем студента в стазис
    for p in student.parameters():
        p.data.requires_grad = False
    
    #обнуляем веса сегментационной головы
    validation_head.reset()

    val_optim = torch.optim.AdamW(validation_head.parameters(), lr = 1e-3)
    validation_head.cuda()
    
    val_loss = 0
    best_IoU = 0 

    total_batches = val_epochs * len(validation_loader)
    with tqdm(total=total_batches, desc='Validation', unit='batch') as pbar:
        for _ in range(val_epochs):
            IoU = 0
            for batch in validation_loader:
                image, target = batch
                image = image.cuda()
                target = target.cuda()
                encoded = student(image).feature_maps[3]
                
                prediction = validation_head(encoded)

                loss = loss_fn(prediction, target.long())
                loss.backward()
                val_loss += loss / batch_size

                val_optim.step()
                val_optim.zero_grad()

                IoU += compute_miou(prediction, target, num_classes = 2)

                pbar.update(1)
            
            IoU /= len(validation_loader)
            pbar.set_postfix({'IoU': IoU})
            best_IoU = max(best_IoU, IoU)
            

    val_loss /= len(validation_loader) * val_epochs
    

    wdist = 0
    for gs, gt, in zip(student.parameters(), teacher.parameters()):
        wdist += torch.sqrt(((gt.data - gs.data) ** 2).mean())
    for gs, gt, in zip(student_HEAD.parameters(), teacher_HEAD.parameters()):
        wdist += torch.sqrt(((gt.data - gs.data) ** 2).mean())

    student_embeddings = torch.cat(student_embeddings[:25], dim = 0)
    embeddings_pca = metodpisi.fit_transform(student_embeddings)
    pca_labels = torch.arange(0, batch_size).repeat(student_embeddings.shape[0])
    pca_labels += (torch.arange(student_embeddings.shape[0] // 2) * batch_size).repeat_interleave(batch_size*2)

    pca_fig = plot_pca_2d(
        embeddings_pca, 
        pca_labels.tolist(), 
        epoch=epoch,    
        view_markers={'v1': 'o', 'v2': 'x'}  
    )

    print(f'Epoch {epoch}')
    print(f'Train loss {train_loss:.2f}')
    print(f'Validation loss {val_loss:.2f}')
    print(f'Validation IoU {best_IoU:.2f}')
    print(f'Weight distance {wdist:.2f}')
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('Weight distance', wdist, epoch)
    writer.add_scalar('IoU', best_IoU, epoch)
    writer.add_figure('PCA', pca_fig, epoch)
    plt.close()

    scheduler.step(train_loss.detach())
    

Validation: 100%|██████████| 50/50 [00:42<00:00,  1.18batch/s, IoU=tensor(0.8139, device='cuda:0')]


Epoch 0
Train loss 2.75
Validation loss 0.10
Validation IoU 0.82
Weight distance 1.12


Validation: 100%|██████████| 50/50 [00:42<00:00,  1.18batch/s, IoU=tensor(0.8311, device='cuda:0')]


Epoch 1
Train loss 2.77
Validation loss 0.09
Validation IoU 0.83
Weight distance 1.34


Validation: 100%|██████████| 50/50 [00:43<00:00,  1.14batch/s, IoU=tensor(0.8674, device='cuda:0')]


...

Epoch 3
Train loss 2.77
Validation loss 0.09
Validation IoU 0.84
Weight distance 2.06


Validation: 100%|██████████| 50/50 [00:43<00:00,  1.15batch/s, IoU=tensor(0.8826, device='cuda:0')]


Epoch 4
Train loss 2.74
Validation loss 0.09
Validation IoU 0.88
Weight distance 1.97


Validation:  24%|██▍       | 12/50 [00:11<00:35,  1.08batch/s, IoU=tensor(0.2752, device='cuda:0')]


KeyboardInterrupt: 